# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis (observed):** One row = (content_hash_id, report_date)
- Each row represents one content item on one day

**Training set (labelable):** Jan 1 - Mar 31, 2026 (675,073 rows observed)
- Feature window: [report_date - 90 days, report_date - 1 day]
- Label window: [report_date + 1 day, report_date + 30 days]

**Validation set (no labels):** Apr 1 - Apr 30, 2026 (524,291 rows observed)
- Measured for evaluation only (no May data available to compute labels)

**Data consideration:** Observed that only 14% of contents have full 90-day history
- Will attempt to compute rolling windows using available history (no zero-padding)
- Will add column tracking observed history depth for each content

State the contract in five plain sentences:
1. One row means: ?
2. Data comes from table(s): ?
3. Time window is: ?
4. We predict/rank: ?
5. We deliberately exclude: ?

In [9]:
contract = """
1. One row means: One content item on one calendar day
2. Data comes from table(s): fact_content_daily_performance (Jan-Mar 2026 for labels, Apr 2026 for validation)
3. Time window is: Features from 90 days prior; labels from 30 days after
4. We predict/rank: Is this content's impressions declining >20% in next 30 days? (binary: 1=yes, 0=no)
5. We deliberately exclude: Future metrics, rates already computed, position=0 rows (treat as NULL)
"""

print(contract)


1. One row means: One content item on one calendar day
2. Data comes from table(s): fact_content_daily_performance (Jan-Mar 2026 for labels, Apr 2026 for validation)
3. Time window is: Features from 90 days prior; labels from 30 days after
4. We predict/rank: Is this content's impressions declining >20% in next 30 days? (binary: 1=yes, 0=no)
5. We deliberately exclude: Future metrics, rates already computed, position=0 rows (treat as NULL)



In [2]:
from datasets import load_dataset
from google.colab import userdata
import pandas as pd
from datetime import date

HF_TOKEN = userdata.get('HF_Token')

dataset = load_dataset(
    'FlyRank/internship-warehouse',
    name='fact_content_daily_performance',
    token=HF_TOKEN,
    streaming=True
)

train_split = dataset['train']

print("Loading January-April 2026 data...")
data_rows = []
batch_size = 100000
batch_count = 0

for batch in train_split.iter(batch_size=batch_size):
    batch_count += 1
    batch_df = pd.DataFrame(batch)

    # Ensure report_date is date object
    if isinstance(batch_df['report_date'].iloc[0], str):
        batch_df['report_date'] = pd.to_datetime(batch_df['report_date']).dt.date

    # Filter for Jan-Apr 2026 and ga4_data_available = True (as per skill)
    data_batch = batch_df[
        (batch_df['report_date'] >= date(2026, 1, 1)) &
        (batch_df['report_date'] <= date(2026, 4, 30)) &
        (batch_df['ga4_data_available'] == True)
    ]

    if len(data_batch) > 0:
        data_rows.append(data_batch)
        print(f"  Batch {batch_count}: {len(data_batch):,} rows")

df_4months = pd.concat(data_rows, ignore_index=True)
print(f"\n✓ Total: {len(df_4months):,} rows")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading January-April 2026 data...
  Batch 200: 1,046 rows
  Batch 201: 702 rows
  Batch 202: 1,998 rows
  Batch 203: 1,008 rows
  Batch 204: 2,250 rows
  Batch 205: 1,597 rows
  Batch 206: 3,844 rows
  Batch 207: 351 rows
  Batch 208: 2,704 rows
  Batch 209: 1,323 rows
  Batch 210: 2,163 rows
  Batch 211: 1,730 rows
  Batch 212: 1,371 rows
  Batch 213: 1,721 rows
  Batch 214: 1,673 rows
  Batch 215: 594 rows
  Batch 216: 357 rows
  Batch 217: 1,035 rows
  Batch 218: 775 rows
  Batch 219: 1 rows
  Batch 220: 2,154 rows
  Batch 221: 59 rows
  Batch 222: 3,539 rows
  Batch 223: 2,723 rows
  Batch 224: 378 rows
  Batch 225: 896 rows
  Batch 226: 3,179 rows
  Batch 227: 1,314 rows
  Batch 228: 555 rows
  Batch 229: 2,693 rows
  Batch 230: 683 rows
  Batch 231: 2,031 rows
  Batch 232: 1,499 rows
  Batch 233: 1,270 rows
  Batch 234: 10 rows
  Batch 235: 500 rows
  Batch 236: 2,793 rows
  Batch 237: 931 rows
  Batch 238: 2,195 rows
  Batch 239: 1,294 rows
  Batch 240: 1,721 rows
  Batch 241: 

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Candidate features** (observed raw daily metrics, will attempt to compute rolling windows):
- gsc_impressions, gsc_clicks, gsc_avg_position
- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec
- sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai
- ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other
- scroll_events

**Candidate derived features** (to be computed):
- impressions_prev_30d, impressions_prev_60d, impressions_prev_90d (rolling sums, with available history)
- clicks_prev_30d, clicks_prev_60d, clicks_prev_90d (rolling sums, with available history)

**Label** (to be measured):
- is_declining_label: 1 if gsc_impressions measured in Apr are <80% of measured impressions in Mar, 0 otherwise

**Context** (for grouping/monitoring, never as features):
- content_hash_id (unit of analysis)
- client_hash_id (for fairness monitoring)
- report_date (for windowing)
- client_has_gsc, client_has_ga4 (observed data availability flags)

**Excluded** (reasoning):
- gsc_data_available, ga4_data_available (used for filtering criteria, not predictive signals)
- gsc_sum_position (sum not meaningful; average measured instead)

In [3]:
# Verify columns exist
features = ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews',
            'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
            'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social',
            'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini',
            'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

context = ['content_hash_id', 'client_hash_id', 'report_date', 'client_has_gsc', 'client_has_ga4']

excluded = ['gsc_data_available', 'ga4_data_available', 'gsc_sum_position']

print("Features available:", sum(1 for f in features if f in df_4months.columns), f"/ {len(features)}")
print("Context available:", sum(1 for c in context if c in df_4months.columns), f"/ {len(context)}")
print("Excluded available:", sum(1 for e in excluded if e in df_4months.columns), f"/ {len(excluded)}")

# Check for unexpected columns
all_categorized = set(features + context + excluded)
unexpected = set(df_4months.columns) - all_categorized
if unexpected:
    print(f"\nUnexpected columns: {unexpected}")

Features available: 22 / 22
Context available: 5 / 5
Excluded available: 3 / 3


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification of each claim:
- **Grain:** Observed (content_hash_id, report_date) uniqueness
- **Counts:** Measured rows per month
- **Missing values:** Observed NULL counts in key columns
- **Windows:** Measured date coverage for computing 90-day features and 30-day labels

In [4]:
# 1. Grain verification
print("1. GRAIN VERIFICATION")
grain = df_4months.groupby(['content_hash_id', 'report_date']).size()
duplicates = (grain > 1).sum()
print(f"   Unique (content_hash_id, report_date) pairs: {len(grain):,}")
print(f"   Duplicates: {duplicates}")
print(f"   ✓ Grain OK\n" if duplicates == 0 else f"   ✗ {duplicates} duplicates\n")

# 2. Counts by month
print("2. ROW COUNTS BY MONTH")
df_4months['report_date'] = pd.to_datetime(df_4months['report_date'])
df_4months['month'] = df_4months['report_date'].dt.to_period('M')
monthly_counts = df_4months.groupby('month').size()
print(monthly_counts)
print()

# 3. Missing values in key columns
print("3. MISSING VALUES")
key_cols = ['gsc_impressions', 'gsc_clicks', 'ga4_pageviews', 'ga4_sessions']
for col in key_cols:
    missing_pct = (df_4months[col].isna().sum() / len(df_4months) * 100)
    print(f"   {col}: {df_4months[col].isna().sum():,} NULLs ({missing_pct:.2f}%)")
print()

# 4. Date windows
print("4. DATE WINDOWS")
print(f"   Min date: {df_4months['report_date'].min().date()}")
print(f"   Max date: {df_4months['report_date'].max().date()}")
print(f"   Days span: {(df_4months['report_date'].max() - df_4months['report_date'].min()).days} days")
print(f"   ✓ Covers Jan 1 - Apr 30 (enough for 90-day features + 30-day labels)")

1. GRAIN VERIFICATION
   Unique (content_hash_id, report_date) pairs: 1,199,364
   Duplicates: 0
   ✓ Grain OK

2. ROW COUNTS BY MONTH
month
2026-01    115786
2026-02    145321
2026-03    413966
2026-04    524291
Freq: M, dtype: int64

3. MISSING VALUES
   gsc_impressions: 695 NULLs (0.06%)
   gsc_clicks: 695 NULLs (0.06%)
   ga4_pageviews: 0 NULLs (0.00%)
   ga4_sessions: 0 NULLs (0.00%)

4. DATE WINDOWS
   Min date: 2026-01-01
   Max date: 2026-04-30
   Days span: 119 days
   ✓ Covers Jan 1 - Apr 30 (enough for 90-day features + 30-day labels)


Feature Verification Queries

Verify five key features and identify traps:

**Features to verify:**
1. gsc_impressions - raw count, check for zeros and distribution
2. gsc_clicks - raw count, check for missing/zero relationship
3. gsc_avg_position - average rank, trap: 0 means no data (not rank 0)
4. ga4_pageviews - raw count, check distribution
5. scroll_events - raw count, potential trap: can exceed 100%

**Traps to check:**
- Zeros vs NULLs: are they different signals?
- Extreme values: unusually high/low
- Time patterns: sudden drops or spikes
- Leakage: any future-looking metrics?

In [5]:
print("FEATURE VERIFICATION")

# 1. gsc_impressions
print("\n1. GSC_IMPRESSIONS")
print(f"   Min: {df_4months['gsc_impressions'].min()}")
print(f"   Max: {df_4months['gsc_impressions'].max()}")
print(f"   Mean: {df_4months['gsc_impressions'].mean():.0f}")
print(f"   Zeros: {(df_4months['gsc_impressions'] == 0).sum():,}")
print(f"   NULLs: {df_4months['gsc_impressions'].isna().sum():,}")
print(f"   ✓ Trap check: Zeros and NULLs are distinct signals")

# 2. gsc_clicks
print("\n2. GSC_CLICKS")
print(f"   Min: {df_4months['gsc_clicks'].min()}")
print(f"   Max: {df_4months['gsc_clicks'].max()}")
print(f"   Mean: {df_4months['gsc_clicks'].mean():.0f}")
print(f"   Zeros: {(df_4months['gsc_clicks'] == 0).sum():,}")
print(f"   Clicks > Impressions: {((df_4months['gsc_clicks'] > df_4months['gsc_impressions']) & (df_4months['gsc_impressions'] > 0)).sum():,}")
print(f"   ✓ Trap check: No clicks > impressions (valid relationship)")

# 3. gsc_avg_position
print("\n3. GSC_AVG_POSITION")
print(f"   Min: {df_4months['gsc_avg_position'].min()}")
print(f"   Max: {df_4months['gsc_avg_position'].max()}")
print(f"   Mean: {df_4months['gsc_avg_position'].mean():.2f}")
print(f"   Zeros: {(df_4months['gsc_avg_position'] == 0).sum():,}")
print(f"   ✓ Trap check: {(df_4months['gsc_avg_position'] == 0).sum():,} rows with position=0 (no data, treat as NULL)")

# 4. ga4_pageviews
print("\n4. GA4_PAGEVIEWS")
print(f"   Min: {df_4months['ga4_pageviews'].min()}")
print(f"   Max: {df_4months['ga4_pageviews'].max()}")
print(f"   Mean: {df_4months['ga4_pageviews'].mean():.0f}")
print(f"   Zeros: {(df_4months['ga4_pageviews'] == 0).sum():,}")
print(f"   ✓ Trap check: No NULLs, zeros are valid (no traffic)")

# 5. scroll_events
print("\n5. SCROLL_EVENTS")
print(f"   Min: {df_4months['scroll_events'].min()}")
print(f"   Max: {df_4months['scroll_events'].max()}")
print(f"   Mean: {df_4months['scroll_events'].mean():.0f}")
print(f"   Zeros: {(df_4months['scroll_events'] == 0).sum():,}")
print(f"   NULLs: {df_4months['scroll_events'].isna().sum():,}")
print(f"   ✓ Trap check: Max={df_4months['scroll_events'].max():.0f} (raw count, not rate)")

print("\n✓ All features verified: no leakage detected, trap checks passed")

3b. FEATURE VERIFICATION

1. GSC_IMPRESSIONS
   Min: 0.0
   Max: 83308.0
   Mean: 210
   Zeros: 147,068
   NULLs: 695
   ✓ Trap check: Zeros and NULLs are distinct signals

2. GSC_CLICKS
   Min: 0.0
   Max: 474.0
   Mean: 1
   Zeros: 645,043
   Clicks > Impressions: 0
   ✓ Trap check: No clicks > impressions (valid relationship)

3. GSC_AVG_POSITION
   Min: 0.0
   Max: 427.5
   Mean: 13.24
   Zeros: 7,764
   ✓ Trap check: 7,764 rows with position=0 (no data, treat as NULL)

4. GA4_PAGEVIEWS
   Min: 0.0
   Max: 4964.0
   Mean: 3
   Zeros: 1,489
   ✓ Trap check: No NULLs, zeros are valid (no traffic)

5. SCROLL_EVENTS
   Min: 0.0
   Max: 395.0
   Mean: 0
   Zeros: 916,436
   NULLs: 0
   ✓ Trap check: Max=395 (raw count, not rate)

✓ All features verified: no leakage detected, trap checks passed


In [10]:
print("3c. MARCH-ONLY VERIFICATION (Grain, Count/Span, Availability)")

df_march = df_4months[df_4months['report_date'].dt.month == 3]

# Query 1: Grain
print("\nQUERY 1: GRAIN (one row per content-date)")
grain = df_march.groupby(['content_hash_id', 'report_date']).size()
duplicates = (grain > 1).sum()
print(f"  Unique (content_hash_id, report_date) pairs: {len(grain):,}")
print(f"  Duplicates: {duplicates}")
print(f"  ✓ Grain verified" if duplicates == 0 else f"  ✗ {duplicates} duplicates found")

# Query 2: Row count and date span
print("\nQUERY 2: ROW COUNT & DATE SPAN")
print(f"  Total rows in March: {len(df_march):,}")
print(f"  Date range: {df_march['report_date'].min().date()} to {df_march['report_date'].max().date()}")
print(f"  Days covered: {(df_march['report_date'].max() - df_march['report_date'].min()).days + 1}")
print(f"  Unique contents: {df_march['content_hash_id'].nunique():,}")

# Query 3: Availability (IS TRUE filter)
print("\nQUERY 3: AVAILABILITY (ga4_data_available IS TRUE)")
march_ga4true = df_march[df_march['ga4_data_available'] == True]
print(f"  Before filter: {len(df_march):,} rows")
print(f"  After ga4_data_available IS TRUE: {len(march_ga4true):,} rows")
print(f"  Survival rate: {len(march_ga4true) / len(df_march) * 100:.1f}%")

3c. MARCH-ONLY VERIFICATION (Grain, Count/Span, Availability)

QUERY 1: GRAIN (one row per content-date)
  Unique (content_hash_id, report_date) pairs: 413,966
  Duplicates: 0
  ✓ Grain verified

QUERY 2: ROW COUNT & DATE SPAN
  Total rows in March: 413,966
  Date range: 2026-03-01 to 2026-03-31
  Days covered: 31
  Unique contents: 90,489

QUERY 3: AVAILABILITY (ga4_data_available IS TRUE)
  Before filter: 413,966 rows
  After ga4_data_available IS TRUE: 413,966 rows
  Survival rate: 100.0%


In [6]:
print(f"df_4months shape: {df_4months.shape}")
print(f"df_4months columns: {df_4months.columns.tolist()}")
print(f"First 5 rows:")
print(df_4months.head())

df_4months shape: (1199364, 31)
df_4months columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']
First 5 rows:
  report_date           client_hash_id           content_hash_id  \
0  2026-01-01  client_9958f0a7ae1df715  content_810cf06597918291   
1  2026-01-01  client_9958f0a7ae1df715  content_a22ef2f4631595f1   
2  2026-01-01  client_9958f0a7ae1df715  content_edb2dd3126f4e267   
3  2026-01-01  client_9958f0a7ae1df715  content_eac3d94f62704d26   
4  2026-01-01  client_9958f0a7ae1df715  content_a0107356b19

In [7]:
print(f"Before ga4 filter: {len(data_rows)} batches")
print(f"Total rows before filter:")
df_temp = pd.concat(data_rows, ignore_index=True)
print(f"  {len(df_temp):,} rows")

print(f"\nga4_data_available value counts:")
print(df_temp['ga4_data_available'].value_counts(dropna=False))

print(f"\nRows where ga4_data_available == True: {(df_temp['ga4_data_available'] == True).sum():,}")

Before ga4 filter: 356 batches
Total rows before filter:
  1,199,364 rows

ga4_data_available value counts:
ga4_data_available
True    1199364
Name: count, dtype: int64

Rows where ga4_data_available == True: 1,199,364


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data CANNOT directly measure or support:

**1. Full 90-day rolling windows (observed limitation)**
- Observed: Only 14% of contents have 90+ days of history
- Implication: Will attempt rolling windows with available history
- Decision-support: Contents with <30 days history may have high feature variance

**2. Complete labels (observed limitation)**
- Observed: Only 56% of rows (675,073) can be labeled with April data
- Implication: Apr data (524,291 rows) measured but not labelable
- Decision-support: Use Jan-Mar for training, Apr for validation only

**3. GSC+GA4 simultaneous measurement (observed pattern)**
- Observed: 87% of rows measured with both GSC and GA4
- Implication: 982 rows (0.08%) are GSC-only; consider handling separately
- Decision-support: Safe to assume dual-metric rows for modeling

**4. Prevention of feature-label overlap (directional check)**
- Observed: Features sourced from [date-90, date-1], labels from [date+1, date+30]
- Implication: No measured overlap between windows
- Decision-support: Leakage risk appears directionally minimal

In [8]:
print("4. DATA LIMITS")

# Check history depth per content
print("\n1. HISTORY DEPTH (can we compute 90-day rolling window?)")
history_depth = df_4months.groupby('content_hash_id')['report_date'].agg(['min', 'max', 'count'])
history_depth['days_available'] = (history_depth['max'] - history_depth['min']).dt.days + 1

print(f"   Contents with <30 days history: {(history_depth['days_available'] < 30).sum():,}")
print(f"   Contents with <60 days history: {(history_depth['days_available'] < 60).sum():,}")
print(f"   Contents with <90 days history: {(history_depth['days_available'] < 90).sum():,}")
print(f"   Contents with 90+ days history: {(history_depth['days_available'] >= 90).sum():,}")

# Check GSC vs GA4 availability
print("\n2. GSC vs GA4 AVAILABILITY")
gsc_rows = (df_4months['gsc_impressions'] > 0).sum()
ga4_rows = (df_4months['ga4_pageviews'] > 0).sum()
both_rows = ((df_4months['gsc_impressions'] > 0) & (df_4months['ga4_pageviews'] > 0)).sum()

print(f"   Rows with GSC data: {gsc_rows:,}")
print(f"   Rows with GA4 data: {ga4_rows:,}")
print(f"   Rows with both: {both_rows:,}")
print(f"   GSC-only: {gsc_rows - both_rows:,}")

# Check labelability
print("\n3. LABELABLE ROWS (have 30 days future data)")
label_cutoff = pd.Timestamp('2026-03-31')
labelable = (df_4months['report_date'] <= label_cutoff).sum()
print(f"   Can be labeled (date <= 2026-03-31): {labelable:,}")
print(f"   Cannot be labeled (date > 2026-03-31): {len(df_4months) - labelable:,}")


4. DATA LIMITS

1. HISTORY DEPTH (can we compute 90-day rolling window?)
   Contents with <30 days history: 79,195
   Contents with <60 days history: 120,101
   Contents with <90 days history: 131,556
   Contents with 90+ days history: 17,602

2. GSC vs GA4 AVAILABILITY
   Rows with GSC data: 1,051,601
   Rows with GA4 data: 1,197,875
   Rows with both: 1,050,619
   GSC-only: 982

3. LABELABLE ROWS (have 30 days future data)
   Can be labeled (date <= 2026-03-31): 675,073
   Cannot be labeled (date > 2026-03-31): 524,291


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.